In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin
from datetime import datetime
import pickle
import os

parquet_file = "../../data/00-newspaper_data/crawler/jornada/articles_morelos.parquet"

# Base URL for paginated pages
base_url = "https://www.lajornadamorelos.mx/page/{page_number}/"

# Month mapping for date conversion
month_mapping = {
    "Ene": "01", "Feb": "02", "Mar": "03", "Abr": "04", "May": "05", "Jun": "06",
    "Jul": "07", "Ago": "08", "Sep": "09", "Oct": "10", "Nov": "11", "Dic": "12"
}

# File to save processed URLs
processed_urls_file = "../../data/00-newspaper_data/crawler/jornada/processed_urls.pkl"

# Load processed URLs if the file exists
if os.path.exists(processed_urls_file):
    with open(processed_urls_file, 'rb') as f:
        processed_urls = pickle.load(f)
else:
    processed_urls = set()


def clean_main_text(text):
    """
    Remove unwanted phrases from the main text.
    """
    unwanted_phrases = [
        "LA JORNADA MORELOS",
        "La Jornada Morelos (Prohibida la reproducción total o parcial del contenido de esta publicación, por cualquier medio, sin permiso expreso de los editores)"
    ]
    for phrase in unwanted_phrases:
        text = text.replace(phrase, "")
    return text.strip()

def is_relevant_url(url):
    """
    Check if the URL matches the pattern of an article URL on lajornadamorelos.mx.
    Exclude URLs containing 'category' or 'author', or ending with a date or numeric ID.
    """
    pattern = r"https://www\.lajornadamorelos\.mx/[^/]+/[^/]+"
    exclude_terms = ["category", "author", 'page']
    date_only_pattern = r"https://www\.lajornadamorelos\.mx/\d{4}/\d{2}/$"
    topic_with_number_pattern = r"https://www\.lajornadamorelos\.mx/[^/]+/\d+/$"

    if (
        re.match(pattern, url) 
        and not any(term in url for term in exclude_terms) 
        and not re.match(date_only_pattern, url) 
        and not re.match(topic_with_number_pattern, url)
    ):
        return True
    return False

def convert_date(date_text):
    """
    Convert date from format 'Sep 19 2024' to '2024/09/19' using month mapping.
    """
    print(date_text)
    try:
        parts = date_text.replace(",", "").split()
        month = month_mapping[parts[0]]
        day = parts[1]
        year = parts[2]
        return f"{year}/{month}/{day.zfill(2)}"
    except (IndexError, KeyError):
        print(f"Date format error for '{date_text}'")
        return None

def extract_article_content(url):
    """
    Extract the title, main text, and date from a given article URL.
    """
    article_data = {
        "url": url,
        "title": None,
        "main_text": None,
        "date": None
    }

    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # Extract title
        title_element = soup.find("h1", class_="title single")
        if title_element:
            title_element = title_element.find("a")
            article_data["title"] = title_element.get_text(strip=True) if title_element else None
        #print(article_data["title"])

        # Extract main text from <p> tags
        parags = soup.find_all('p')
        if parags: 
            paragraphs = [p.get_text(strip=True) for p in parags]
            main_text = " ".join(paragraphs) if paragraphs else None
            article_data["main_text"] = clean_main_text(main_text) if main_text else None
        #print(article_data["main_text"])

        # Extract and convert date
        date_element = soup.find("span", class_="mg-blog-date")
        article_data["date"] = date_element.get_text(strip=True) if date_element else None
        #print(article_data["date"])
        if date_element:
            date_text = date_element.get_text(strip=True)
            article_data["date"] = convert_date(date_text)

    except requests.exceptions.RequestException as e:
        print(f"Error accessing {url}: {e}")

    return article_data

def find_articles_on_page(page_number):
    """
    Get all relevant article URLs from a specific page.
    """
    url = base_url.format(page_number=page_number)
    articles = []

    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # Find and filter relevant article URLs
        for link in soup.find_all("a", href=True):
            href = urljoin(url, link['href'])
            if "lajornadamorelos" in href and is_relevant_url(href):
                articles.append(href)

    except requests.exceptions.RequestException as e:
        print(f"Error accessing {url}: {e}")

    return list(set(articles))  # Remove duplicate URLs

def scrape_articles(max_pages=10):
    """
    Loop over specified number of pages, extract article data, and save to DataFrame.
    Only process new URLs that haven't been processed before.
    """
    all_articles = []

    for page_number in range(1, max_pages + 1):
        print(f"Extracting articles from page {page_number}")
        article_urls = find_articles_on_page(page_number)

        for url in article_urls:
            if url not in processed_urls:
                article_data = extract_article_content(url)
                if article_data["title"] and article_data["main_text"] and article_data["date"]:
                    all_articles.append(article_data)
                    processed_urls.add(url)  # Mark URL as processed
                    print(f"Extracted data from {url}")

        # Save processed URLs after each page
        with open(processed_urls_file, 'wb') as f:
            pickle.dump(processed_urls, f)

     # Convert collected data to DataFrame
    if all_articles:
            new_articles_df = pd.DataFrame(all_articles)
            if os.path.exists(parquet_file):
                existing_articles = pd.read_parquet(parquet_file)
            else:
                existing_articles = pd.DataFrame()
            existing_articles = pd.concat([existing_articles, new_articles_df]).drop_duplicates(subset="url", keep="last")
            existing_articles.to_parquet(parquet_file, compression="gzip")
            print(f"Data saved to '{parquet_file}' after page {page_number}")

# Example usage
scrape_articles(max_pages=2415)
print("Final data collection completed.")


Extracting articles from page 1
Nov 18, 2024
Extracted data from https://www.lajornadamorelos.mx/lo-mas-visto/morelos-santuario-de-trabajadores-migrantes/
Nov 18, 2024
Extracted data from https://www.lajornadamorelos.mx/sociedad/vamos-por-la-consolidacion-en-unidad-y-fortaleza-de-morena-suarez-maldonado/
Nov 18, 2024
Extracted data from https://www.lajornadamorelos.mx/sociedad/inicio-el-xxv-encuentro-de-pueblos-negros/
Nov 18, 2024
Extracted data from https://www.lajornadamorelos.mx/municipios/zacatepec/en-zacatepec-ya-tienen-un-mercado-digno-pero-aun-requiere-ajustes/
Nov 18, 2024
Extracted data from https://www.lajornadamorelos.mx/opinion/cibernetica-politica-y-sociedad-34/
Nov 18, 2024
Extracted data from https://www.lajornadamorelos.mx/opinion/democracia-real-90/
Nov 18, 2024
Extracted data from https://www.lajornadamorelos.mx/sociedad/exigen-abogados-rendicion-de-cuentas-por-viaje-a-holanda-de-magistradas-del-tja/
Oct 4, 2022
Extracted data from https://www.lajornadamorelos.mx/soc

In [20]:
asss = pd.read_parquet(parquet_file)
asss

,url,title,main_text,date
0,https://www.lajornadamorelos.mx/sociedad/cierr...,Cierran filas Ejecutivo y Cuernavaca por la se...,Con el objetivo de seguir fortaleciendo y coor...,2024/11/14
1,https://www.lajornadamorelos.mx/opinion/salud-...,SALUD PARA TODOS,¿Cómo participa la familia viviendo con hipert...,2024/11/14
2,https://www.lajornadamorelos.mx/uaem/la-uaem-e...,La UAEM está lista para defender su autonomía:...,La rectora de la Universidad Autónoma del Esta...,2024/11/14
3,https://www.lajornadamorelos.mx/sociedad/morel...,Morelos será sede del Foro Mundial de Gastrono...,La gobernadora Margarita González Saravia enca...,2024/11/14
4,https://www.lajornadamorelos.mx/opinion/fosas-...,"Fosas clandestinas, personas desaparecidas y D...",La existencia de fosas clandestinas en el país...,2024/11/14
5,https://www.lajornadamorelos.mx/zafra/la-auton...,"La autonomía universitaria, más que embates, r...","ZAFRA Haz tu Denuncia, queja, sugerencia ...",2024/11/14
6,https://www.lajornadamorelos.mx/sociedad/confl...,Conflictos obrero-patronales se resolverán por...,Ciudad de México.Las autoridades laborales die...,2022/10/04
7,https://www.lajornadamorelos.mx/opinion/4t-la-...,4T LA AGENDA POLÍTICO-LEGAL,LA INJUSTICIA DE LA JUSTICA. Más de doscientas...,2024/11/14
8,https://www.lajornadamorelos.mx/opinion/pueblo...,Pueblos indígenas y sindicatos en pie de lucha...,La defensa de los derechos humanos es tarea de...,2024/11/14
9,https://www.lajornadamorelos.mx/opinion/algo-c...,Algo como una fruta: Anecdotario estacional de...,Hace muchos años soñé que entraba a una habita...,2024/11/14
